# 07 – SoftDCD & CombinedSoftMin Loss Comparison

## Hypothesis

**SoftDCDLoss** (Soft Determinantal Coverage Diversity) explicitly rewards prediction diversity and coverage simultaneously by penalising both under-coverage and redundant predictions via a soft-determinant term. **CombinedSoftMinLoss** blends Chamfer distance with PowerSoftMin, adding a direct diversity bonus through the Chamfer symmetric matching. Both might outperform pure PowerSoftMin by providing richer training signal that prevents mode collapse.

## Experiment

We compare four training conditions on the CLEVR set-prediction task:

| Label | Loss | Notes |
|---|---|---|
| `PM3 τ=0.10` | `PowerSoftMinLoss(τ=0.10, p=3)` | Baseline (sharp) |
| `PM3 τ=0.12` | `PowerSoftMinLoss(τ=0.12, p=3)` | Baseline (softer) |
| `SoftDCD` | `SoftDCDLoss()` | Coverage + diversity |
| `CombinedSM` | `CombinedSoftMinLoss()` | Chamfer + PowerSoftMin blend |

Entropy and PCA snapshots are captured at epochs 0, 5, 15, 35, 49.

In [1]:
import sys
sys.path.insert(0, '.')
import clevr_utils as cu
from influencerformer.losses import PowerSoftMinLoss

try:
    from influencerformer.losses import SoftDCDLoss
    HAS_SOFTDCD = True
except ImportError:
    print('WARNING: SoftDCDLoss not found in influencerformer.losses – skipping that condition.')
    HAS_SOFTDCD = False

try:
    from influencerformer.losses import CombinedSoftMinLoss
    HAS_COMBINEDSM = True
except ImportError:
    print('WARNING: CombinedSoftMinLoss not found in influencerformer.losses – skipping that condition.')
    HAS_COMBINEDSM = False

print(f'Device: {cu.DEVICE}')

Device: cuda


In [2]:
N_EPOCHS = 50
SNAPSHOT_EPOCHS = {0, 5, 15, 35, 49}

In [3]:
all_results = {}

conditions = [
    ('PM3 τ=0.12', PowerSoftMinLoss(temperature=0.12, power=3.0), True),
    ('SoftDCD',    SoftDCDLoss()    if HAS_SOFTDCD    else None, HAS_SOFTDCD),
    ('CombinedSM', CombinedSoftMinLoss() if HAS_COMBINEDSM else None, HAS_COMBINEDSM),
]

for name, loss_fn, available in conditions:
    if not available:
        print(f'Skipping {name} (loss not available).')
        continue
    m, s = cu.train_and_monitor(
        loss_fn,
        name,
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        tau_for_entropy=0.12,
    )
    all_results[name] = (m, s)

/global/homes/d/danieltm/.conda/envs/influencer/lib/python3.11/site-packages/torch/nn/modules/transformer.py:296: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343970094/work/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


  [PM3 τ=0.12] epoch   0 | loss=10.5985 dist=1.6252 div=1.487 H=0.224 gd=0.739 lr=3.00e-04
  [PM3 τ=0.12] epoch   1 | loss=6.9616 dist=1.5249 div=1.538 H=0.204 gd=0.591 lr=3.00e-04
  [PM3 τ=0.12] epoch   2 | loss=5.6338 dist=1.4429 div=1.611 H=0.211 gd=0.575 lr=3.00e-04
  [PM3 τ=0.12] epoch   3 | loss=4.6823 dist=1.3842 div=1.708 H=0.204 gd=0.532 lr=3.00e-04
  [PM3 τ=0.12] epoch   4 | loss=4.0530 dist=1.3388 div=1.774 H=0.197 gd=0.676 lr=3.00e-04
  [PM3 τ=0.12] epoch   5 | loss=3.6125 dist=1.3050 div=1.798 H=0.195 gd=0.659 lr=3.00e-04
  [PM3 τ=0.12] epoch   6 | loss=3.2976 dist=1.2851 div=1.823 H=0.190 gd=0.495 lr=3.00e-04
  [PM3 τ=0.12] epoch   7 | loss=3.0575 dist=1.2619 div=1.862 H=0.188 gd=0.562 lr=3.00e-04
  [PM3 τ=0.12] epoch   8 | loss=2.8828 dist=1.2413 div=1.869 H=0.185 gd=0.526 lr=3.00e-04
  [PM3 τ=0.12] epoch   9 | loss=2.7403 dist=1.2224 div=1.919 H=0.180 gd=0.565 lr=3.00e-04
  [PM3 τ=0.12] epoch  10 | loss=2.6335 dist=1.2117 div=1.912 H=0.180 gd=0.520 lr=3.00e-04
  [PM3 τ=

In [ ]:
cu.plot_monitoring(all_results)

# PCA snapshot comparison: SoftDCD vs PM3 τ=0.12
keys_to_compare = [k for k in ['SoftDCD', 'PM3 τ=0.12'] if k in all_results]
if len(keys_to_compare) == 2:
    cu.plot_pca_snapshots(
        {k: all_results[k] for k in keys_to_compare}
    )
else:
    print(f'PCA comparison skipped – available keys: {list(all_results.keys())}')

In [ ]:
cu.summary_table(all_results)